# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzaibazhar895-bit/FlyRank_ai_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Pages with high impressions but low clicks, poor average position, and low engagement are prioritized for review.

The goal is to identify content that is visible in search results but may not be converting that visibility into traffic.

Reason Codes:

CTR_FIX → High impressions but low clicks

POSITION_FIX → Poor average position

LOW_ENGAGEMENT → Low session counts

REVIEW_PRIORITY → Multiple weak signals combined

Signal Check 1: CTR vs Position

Hypothesis: Pages with worse average positions should generally receive fewer clicks.

Verdict: CONFIRMED

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import os
from google.colab import userdata

hf_token = userdata.get("Flyrank_ai_Intern")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT
CASE
    WHEN gsc_avg_position <= 10 THEN 'Top 10'
    WHEN gsc_avg_position <= 20 THEN '11-20'
    ELSE '20+'
END AS position_bucket,
COUNT(*) AS n,
AVG(gsc_clicks) AS avg_clicks
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY 1
ORDER BY 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬──────────────────────┐
│ position_bucket │    n    │      avg_clicks      │
│     varchar     │  int64  │        double        │
├─────────────────┼─────────┼──────────────────────┤
│ 11-20           │  519223 │  0.17805259012023736 │
│ 20+             │ 7138671 │ 0.010940131573509971 │
│ Top 10          │ 2183484 │   0.2982778898311139 │
└─────────────────┴─────────┴──────────────────────┘



Signal Check 2: Impressions vs Sessions

Hypothesis: Pages with higher impressions should generally receive more sessions.

Verdict: CONFIRMED

In [5]:
con.sql(f"""
SELECT
CASE
    WHEN gsc_impressions < 100 THEN 'Low'
    WHEN gsc_impressions < 1000 THEN 'Medium'
    ELSE 'High'
END AS impression_bucket,
COUNT(*) AS n,
AVG(ga4_sessions) AS avg_sessions
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
GROUP BY 1
ORDER BY 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬────────┬────────────────────┐
│ impression_bucket │   n    │    avg_sessions    │
│      varchar      │ int64  │       double       │
├───────────────────┼────────┼────────────────────┤
│ High              │  14821 │  8.174212266378786 │
│ Low               │ 233195 │ 2.3740131649477907 │
│ Medium            │ 165950 │  3.766495932509792 │
└───────────────────┴────────┴────────────────────┘



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Scoring Logic

Higher score means higher review priority.

The score increases when:

1. Impressions are high
2. Clicks are low
3. Average position is poor
4. Sessions are low
Action Label: Review Content

Reason Code: CTR_FIX

In [7]:
import os

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = con.sql(f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE ga4_data_available IS TRUE
GROUP BY content_hash_id
""").df()

df["score"] = (
    (df["gsc_impressions"] / 100)
    - df["gsc_clicks"]
    + (df["gsc_avg_position"] / 10)
    - (df["ga4_sessions"] / 10)
)

df["reason_code"] = "CTR_FIX"
df["action"] = "Review Content"

df = df.sort_values("score", ascending=False)

# Create the directory if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully")

df.head(10)

CSV written successfully


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,score,reason_code,action
47667,content_e8a52cf3d5988c07,244931.0,669.0,15.008339,891.0,1692.710834,CTR_FIX,Review Content
46899,content_44f34c0a90047651,168160.0,15.0,7.324954,37.0,1663.632495,CTR_FIX,Review Content
2023,content_8d7d99f109e19aa2,181942.0,286.0,2.568135,164.0,1517.276814,CTR_FIX,Review Content
2030,content_0e03de7680314cd5,221310.0,720.0,2.675217,465.0,1446.867522,CTR_FIX,Review Content
740,content_36e53e9c707674fc,194579.0,242.0,32.766674,2603.0,1446.766667,CTR_FIX,Review Content
2028,content_4ffe18112a5642e3,186983.0,586.0,2.331060,364.0,1247.663106,CTR_FIX,Review Content
3038,content_3df3f32f3fd58dea,137891.0,194.0,23.251180,347.0,1152.535118,CTR_FIX,Review Content
48213,content_df47d1b976106de4,124727.0,158.0,24.123242,310.0,1060.682324,CTR_FIX,Review Content
48880,content_82e35c4845e6c391,102903.0,54.0,21.904910,59.0,971.320491,CTR_FIX,Review Content
48290,content_5e1c049f62e33b11,117698.0,166.0,18.113831,473.0,965.491383,CTR_FIX,Review Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Content                  | Action         | Reason                                                      | Confidence | What would make it wrong                                        |
| ------------------------ | -------------- | ----------------------------------------------------------- | ---------- | --------------------------------------------------------------- |
| content_44f34c0a90047651 | Review Content | High visibility but weak conversion signals                 | Medium     | Seasonal traffic shifts or incomplete tracking data             |
| content_e8a52cf3d5988c07 | Review Content | CTR improvement opportunity                                 | Medium     | Recent content updates not yet reflected in performance         |
| content_f43118e089ecc69a | Review Content | Weak engagement relative to visibility                      | Medium     | Search demand declined for reasons unrelated to content quality |
| content_8d7d99f109e19aa2 | Review Content | Low traffic capture despite visibility                      | Medium     | Temporary ranking volatility affected results                   |
| content_36e53e9c707674fc | Review Content | Ranking and engagement improvement opportunity              | Medium     | The page serves a niche audience with naturally low engagement  |
| content_0e03de7680314cd5 | Review Content | Poor conversion of impressions into sessions                | Medium     | GA4 tracking may be incomplete for this content                 |
| content_82e35c4845e6c391 | Review Content | Search visibility exists but traffic remains weak           | Medium     | External events temporarily reduced user interest               |
| content_e241d6415ac9e534 | Review Content | Possible CTR optimization candidate                         | Medium     | Search intent may not match the content's purpose               |
| content_471d9cabce329a66 | Review Content | Low engagement signals compared to visibility               | Medium     | Performance may recover naturally without intervention          |
| content_3df3f32f3fd58dea | Review Content | Candidate for content review based on multiple weak signals | Medium     | Normal month-to-month variation may explain the pattern         |


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = df.head(10)

top10[[
    "content_hash_id",
    "score",
    "reason_code",
    "action"
]]

,content_hash_id,score,reason_code,action
47667,content_e8a52cf3d5988c07,1692.710834,CTR_FIX,Review Content
46899,content_44f34c0a90047651,1663.632495,CTR_FIX,Review Content
2023,content_8d7d99f109e19aa2,1517.276814,CTR_FIX,Review Content
2030,content_0e03de7680314cd5,1446.867522,CTR_FIX,Review Content
740,content_36e53e9c707674fc,1446.766667,CTR_FIX,Review Content
2028,content_4ffe18112a5642e3,1247.663106,CTR_FIX,Review Content
3038,content_3df3f32f3fd58dea,1152.535118,CTR_FIX,Review Content
48213,content_df47d1b976106de4,1060.682324,CTR_FIX,Review Content
48880,content_82e35c4845e6c391,971.320491,CTR_FIX,Review Content
48290,content_5e1c049f62e33b11,965.491383,CTR_FIX,Review Content


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks

Some pages may appear because of temporary ranking changes rather than actual content problems.

Some pages may have low sessions because of seasonality rather than poor content quality.

Some pages may have incomplete GA4 tracking.

Leakage Check

No future-window information was used.

No label-derived columns were used.

No future outcomes were included in the score.

Only observed March 2026 metrics were used.

The score is based entirely on information available at the decision moment.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows in ranked queue:", len(df))
print("Unique content items:", df["content_hash_id"].nunique())

Rows in ranked queue: 90489
Unique content items: 90489


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.